# 🏭 MES Multi-Agent AI System (Deva & System Agent)
> **Architecture:** General Orchestrator Agent (Deva) ↔ System Data Agent (Read-Only DB)
>
> - **General Agent (Deva)** — understands intent, decides whether DB data is needed, synthesizes final answers.
> - **System Agent** — holds read-only SQL Server / SQLite access, writes T-SQL, executes via tools (`system_tools`), returns raw results.
> - **System Tools** — `[execute_read_only_sql, get_mes_schema_details, list_available_tables]`
> - **No data modification is possible** — only SELECT queries are permitted.


In [ ]:
# CELL 2: Imports, Environment & LLM Initialization (with Groq fallback)
import os
import json
import re
import operator
import pandas as pd
from typing import Annotated, Sequence, TypedDict, Any
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

import litellm
from langchain_litellm import ChatLiteLLM
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

load_dotenv()
litellm.set_verbose = False

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY")

# ── Primary LLMs: Gemini 3.6 Flash ───────────────────────────────────────
orchestrator_llm_primary = ChatLiteLLM(
    model="gemini/gemini-3.6-flash",
    api_key=GEMINI_API_KEY,
)

system_llm_primary = ChatLiteLLM(
    model="gemini/gemini-3.6-flash",
    api_key=GEMINI_API_KEY,
)

# ── Fallback LLMs: Groq (llama-3.3-70b) ──────────────────────────────────
orchestrator_llm_fallback = ChatLiteLLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)

system_llm_fallback = ChatLiteLLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)

# ── Resilient invoke helpers ──────────────────────────────────────────────
_ACTIVE_PROVIDER = {"orchestrator": "gemini", "system": "gemini"}

def invoke_with_fallback(primary_llm, fallback_llm, messages: list, role: str, tools=None):
    global _ACTIVE_PROVIDER
    llm_primary  = primary_llm.bind_tools(tools)  if tools else primary_llm
    llm_fallback = fallback_llm.bind_tools(tools) if tools else fallback_llm

    try:
        result = llm_primary.invoke(messages)
        if _ACTIVE_PROVIDER[role] != "gemini":
            print(f"   [ROUTER] {role.title()} recovered to Gemini.")
        _ACTIVE_PROVIDER[role] = "gemini"
        return result
    except Exception as e:
        err_str = str(e)
        print(f"   [ROUTER] Gemini failed for {role}: {err_str[:120]}")
        print(f"   [ROUTER] Falling back to Groq for {role}...")
        _ACTIVE_PROVIDER[role] = "groq"
        try:
            result = llm_fallback.invoke(messages)
            print(f"   [ROUTER] Groq responded successfully for {role}.")
            return result
        except Exception as e2:
            raise RuntimeError(
                f"Both Gemini and Groq failed for {role}.\n"
                f"Gemini error: {err_str[:200]}\n"
                f"Groq error: {str(e2)[:200]}"
            ) from e2

print("✅ LLMs initialized — Primary: Gemini 3.6 Flash | Fallback: Groq llama-3.3-70b")
print("   Auto-failover enabled: Gemini → Groq (zero data loss)")


In [ ]:
# CELL 3: Database Engine — Read-Only Connection (SQL Server / SQLite fallback)
DB_DRIVER     = os.getenv("DB_DRIVER",                  "ODBC Driver 18 for SQL Server")
DB_SERVER     = os.getenv("DB_SERVER",                  "localhost")
DB_NAME       = os.getenv("DB_NAME",                    "mes_new")
DB_TRUSTED    = os.getenv("DB_TRUSTED_CONNECTION",      "yes")
DB_ENCRYPT    = os.getenv("DB_ENCRYPT",                 "yes")
DB_TRUST_CERT = os.getenv("DB_TRUST_SERVER_CERTIFICATE","yes")

conn_str = (
    f"DRIVER={{{DB_DRIVER}}};SERVER={DB_SERVER};DATABASE={DB_NAME};"
    f"Trusted_Connection={DB_TRUSTED};Encrypt={DB_ENCRYPT};"
    f"TrustServerCertificate={DB_TRUST_CERT};"
)
connection_url = f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}"

engine = None
try:
    engine = create_engine(connection_url, echo=False)
    with engine.connect() as conn:
        db_name_result = conn.execute(text("SELECT DB_NAME() AS db"))
        db_name_val    = db_name_result.fetchone()[0]
    print(f"✅ Connected to SQL Server — Database: [{db_name_val}]")
    print(f"   Server: {DB_SERVER} | Auth: Windows (Trusted)")
except Exception as e:
    print(f"⚠️ SQL Server connection note: {e}")
    print("   System will run in schema catalog / fallback mode.")


In [ ]:
# CELL 4: MES Schema Catalog — Full 108 tables loaded from schema.sql
MES_CATALOG = {
    "Actions": [
        "Id",
        "ActionName",
        "Icon",
        "CssClass"
    ],
    "AlertMaster": [
        "AlertId",
        "AlertType",
        "Severity",
        "Title",
        "Message",
        "Source",
        "SourceId",
        "WorkOrderId",
        "MachineId",
        "IsAcknowledged",
        "AcknowledgedBy",
        "AcknowledgedAt",
        "IsResolved",
        "ResolvedBy",
        "ResolvedAt",
        "ResolutionNotes",
        "CreatedDate",
        "CustomerId",
        "CreatedBy",
        "UpdatedDate",
        "UpdatedBy"
    ],
    "AlternateUnit": [
        "AlternateUnitId",
        "AlternateUnitName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "AssetType": [
        "AssetTypeId",
        "AssetTypeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "AssetTypeTesting": [
        "AssetTypeId",
        "AssetTypeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "BinMaster": [
        "BinId",
        "BinName",
        "BinCode",
        "MachineId",
        "Capacity",
        "IsActive",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "QRCodeUrl",
        "MachineName"
    ],
    "BOMLine": [
        "BOMLineId",
        "BOMId",
        "ComponentId",
        "Quantity",
        "UOM",
        "Sequence",
        "ScrapPercent",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "BOMMaster": [
        "BOMId",
        "ProductId",
        "BOMVersion",
        "EffectiveFrom",
        "EffectiveTo",
        "Status",
        "Description",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "CapacityAnalysis": [
        "CapacityId",
        "MachineId",
        "Date",
        "ShiftId",
        "AvailableHours",
        "PlannedHours",
        "ActualHours",
        "UtilizationPercent",
        "OverloadFlag",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "Cavity": [
        "CavityId",
        "CavityName",
        "CavityNumber",
        "RawMaterialId",
        "MachineId",
        "FGId",
        "SFGNumber",
        "MessageBox",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "CreatedOn",
        "CompanyId"
    ],
    "ContainerCapacity": [
        "ContainerCapacityId",
        "ContainerCapacityName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "Customer": [
        "CustomerId",
        "CustomerCode",
        "CustomerName",
        "ContactNumber",
        "Email",
        "Address",
        "CityId",
        "StateId",
        "CountryId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "IsActive",
        "DeactivatedDate"
    ],
    "CustomerProductMapping_del": [
        "CustomerId",
        "ProductId",
        "IsActive",
        "CreatedBy",
        "CreatedDate",
        "UpdatedDate",
        "UpdatedBy"
    ],
    "ExciseClassification": [
        "ExciseClassificationId",
        "ExciseClassificationName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "FGRMMapping": [
        "Id",
        "FGCode",
        "RMCode",
        "RMName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "RMQuantityPerKg",
        "RMGrade"
    ],
    "FGSFGMapping": [
        "Id",
        "FGCode",
        "FGName",
        "SFGCode",
        "SFGName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "Type"
    ],
    "FinishedGood": [
        "FGId",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "FGQuantity",
        "BlockedQuantity",
        "NumberOfBags",
        "KGInPerBags",
        "StoreId",
        "UOM",
        "Status",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "QRCodeUrl",
        "BagName",
        "IsHold",
        "Color",
        "IsDispatch",
        "ReasonForRepackaging",
        "RepackDate",
        "IsActive",
        "Id"
    ],
    "GanttSchedule": [
        "GanttId",
        "WorkOrderId",
        "MachineId",
        "StartDate",
        "EndDate",
        "Duration",
        "Color",
        "Progress",
        "Dependencies",
        "CustomerId",
        "ViewType",
        "ScheduledStart",
        "ScheduledEnd",
        "GanttScheduleId",
        "BarLabel",
        "BarStatus",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "Hold": [
        "Id",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "HoldQuantity",
        "UOM",
        "NumberOfBags",
        "KGInPerBags",
        "OutputDate",
        "Status",
        "QRCodeUrl",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "BagName"
    ],
    "HSNCode": [
        "HSNCodeId",
        "HSNCodeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "Inventory": [
        "Id",
        "MaterialCode",
        "Category",
        "BlockedQuantity",
        "QuantityInStock",
        "UpdatedOn",
        "UpdatedBy",
        "CreatedBy",
        "CreatedOn",
        "CompanyId",
        "StockUpdatedReason"
    ],
    "InventoryByLot": [
        "InventoryId",
        "ProductId",
        "LotId",
        "WarehouseCode",
        "LocationCode",
        "Quantity",
        "ReservedQty",
        "LastUpdated",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "ItemCategory": [
        "ItemCategoryId",
        "ItemCategoryCode",
        "ItemCategoryName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "ItemColour": [
        "ItemColourId",
        "ItemColourName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "ItemGroup": [
        "ItemGroupId",
        "ItemGroupCode",
        "ItemGroupName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "ItemMake": [
        "ItemMakeId",
        "ItemMakeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "ItemType": [
        "ItemTypeId",
        "ItemTypeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "JumboPackaging": [
        "Id",
        "RequestId",
        "PackagingType",
        "ScrapQuantity",
        "KGInPerBag",
        "NumberOfBags",
        "Status",
        "Date",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "CreatedOn",
        "CompanyId",
        "QRCodeUrl",
        "StoreId",
        "RejectedQuantity",
        "BagName",
        "Color",
        "UOM",
        "CustomerName",
        "Category",
        "TotalQuantity",
        "IsCompleted"
    ],
    "LotMaster": [
        "LotId",
        "LotNumber",
        "ProductId",
        "ManufactureDate",
        "ExpiryDate",
        "Status",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "Machine": [
        "MachineId",
        "MachineName",
        "MachineCode",
        "MachineType",
        "Model",
        "AverageWeight",
        "PieceOrBox",
        "BoxWeight",
        "Status",
        "Capacity",
        "LastServiceDate",
        "NextServiceDue",
        "ProductionPerDay",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "QRCodeUrl",
        "FGName"
    ],
    "MachineFGMapping": [
        "Id",
        "MachineType",
        "FGCode",
        "FGName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "CycleTime",
        "AverageWeight",
        "NoOfCavities"
    ],
    "MachineMaster": [
        "MachineId",
        "MachineCode",
        "MachineName",
        "MachineType",
        "Location",
        "CapacityPerHour",
        "AvailableHoursPerDay",
        "EfficiencyPercent",
        "Status",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "Capacity",
        "MinCapacity",
        "MaxCapacity",
        "LastMaintDate",
        "NextMaintDate",
        "Shift",
        "Utilization",
        "WorkCenter",
        "Role",
        "Skills"
    ],
    "MachineMaterialMapping": [
        "Id",
        "MachineId",
        "RMCode",
        "SFGCode",
        "FGCode",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "MaintenanceWindow": [
        "MaintenanceId",
        "MachineId",
        "MaintenanceType",
        "StartDate",
        "EndDate",
        "Description",
        "Status",
        "CustomerId",
        "Title",
        "AlertType",
        "DurationHours",
        "WorkCenterCode",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "MaterialDispatch": [
        "Id",
        "SONumber",
        "CustomerCode",
        "NumberOfBags",
        "KGInPerBags",
        "MonthYear",
        "Quantity",
        "UOM",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "QRCodeUrl",
        "Status",
        "DispatchDate",
        "TotalQuantity",
        "LoadedQuantity",
        "UnloadedQuantity",
        "Comments",
        "FinalDispatchQty"
    ],
    "Materials": [
        "MaterialCode",
        "ReferenceNo",
        "MaterialName",
        "PrintName",
        "DisplayName",
        "NameInTally",
        "Description",
        "ItemGroup",
        "ItemCategory",
        "AssetType",
        "ServiceType",
        "DefaultStoreName",
        "MeasurementUnit",
        "AlternateUnit",
        "PackingUnit",
        "PackingSize",
        "BaseConversionRatio",
        "AlternateConversionRatio",
        "SellingPrice",
        "MinimumStock",
        "MaximumStock",
        "ExciseClassification",
        "ItemType",
        "NeckType",
        "Colour",
        "SpareType",
        "PlasticMaterialType",
        "PlasticCategory",
        "ContainerCapacity",
        "Tolerance",
        "DefaultLocation",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "RMGrade"
    ],
    "MaterialTransit": [
        "Id",
        "MaterialCode",
        "Quantity",
        "InTransit",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "MaterialName"
    ],
    "MeasurementUnit": [
        "MeasurementUnitId",
        "MeasurementUnitName",
        "AlternateUnit",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "MPSHeader": [
        "MpsHeaderId",
        "MpsNumber",
        "Description",
        "PlanningPeriod",
        "StartDate",
        "EndDate",
        "Status",
        "ApprovedBy",
        "ApprovedDate",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "MpsMaster": [
        "MpsId",
        "MpsHeaderId",
        "ProductId",
        "PlannedQty",
        "PeriodType",
        "PeriodStart",
        "PeriodEnd",
        "DueDate",
        "DemandSource",
        "SalesOrderRef",
        "Status",
        "IsConverted",
        "WorkOrderId",
        "Notes",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "PlantId",
        "ProductCode",
        "Forecast",
        "PlannedOrders",
        "ProductionQty",
        "Week1",
        "Week2",
        "Week3",
        "Week4",
        "ProjectedStock",
        "MachineName",
        "CapacityMode",
        "CapacityValue",
        "Version",
        "WeeklyPlan"
    ],
    "MPSRawMaterialRequirement": [
        "Id",
        "MpsId",
        "ProductId",
        "MaterialId",
        "RMCode",
        "RMName",
        "QtyPerFG",
        "FGQty",
        "RequiredQty",
        "AvailableStock",
        "ShortageQty",
        "Status",
        "CompanyId",
        "PlantId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "MRP_Demand": [
        "DemandId",
        "MrpRunId",
        "ProductId",
        "DemandSource",
        "SourceReference",
        "RequiredQty",
        "RequiredDate",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "MRP_Result": [
        "ResultId",
        "MrpRunId",
        "ProductId",
        "ActionType",
        "SuggestedQty",
        "SuggestedDate",
        "CurrentStock",
        "ShortageQty",
        "IsAccepted",
        "WorkOrderId",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "MRP_Run": [
        "MrpRunId",
        "RunNumber",
        "RunDate",
        "PlanningHorizonDays",
        "Status",
        "TotalDemands",
        "TotalSuggestions",
        "CompletedDate",
        "RunBy",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "NeckType": [
        "NeckTypeId",
        "NeckTypeName",
        "NeckDescription",
        "ParentNeckName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "NotificationRule": [
        "RuleId",
        "RuleName",
        "AlertType",
        "Condition",
        "Recipients",
        "NotifyEmail",
        "NotifySMS",
        "NotifyInApp",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "UpdatedBy",
        "CreatedDate",
        "UpdatedDate",
        "EscalationMinutes"
    ],
    "OnlineHold": [
        "Id",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "OnlineHoldQuantity",
        "UOM",
        "NumberOfBags",
        "KGInPerBags",
        "OutputDate",
        "Status",
        "StoreId",
        "QRCodeUrl",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "BagName",
        "Color"
    ],
    "OperatorMaster": [
        "OperatorId",
        "OperatorCode",
        "OperatorName",
        "Department",
        "Skill",
        "ShiftId",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "Role",
        "Shift",
        "Skills",
        "Status",
        "Utilization",
        "WorkCenter"
    ],
    "PackagingType": [
        "PackagingTypeId",
        "PackagingTypeName",
        "Capacity",
        "CompanyId",
        "CreatedOn",
        "CreatedBy",
        "UpdatedOn",
        "UpdatedBy"
    ],
    "PlasticCategory": [
        "PlasticCategoryId",
        "PlasticCategoryName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "PlasticMaterialType": [
        "PlasticTypeId",
        "PlasticTypeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "PODetails": [
        "PONumber",
        "Quantity",
        "UnitOfMeasurement",
        "VendorCode",
        "MaterialCode",
        "MaterialName",
        "RMGradeId",
        "Price",
        "Status",
        "RejectedReason",
        "PODate",
        "CreatedBy",
        "CreatedOn",
        "UpdatedOn",
        "UpdatedBy",
        "CompanyId",
        "BillRate"
    ],
    "PriorityMaster": [
        "PriorityId",
        "PriorityCode",
        "PriorityName",
        "PriorityLevel",
        "Color",
        "Description",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "PriorityWeight",
        "SlaHours"
    ],
    "Product_del": [
        "ProductId",
        "ProductName",
        "ProductCode",
        "Category",
        "BrandName",
        "Tags",
        "Description",
        "IsActive",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "Price",
        "StockQuantity",
        "FGId",
        "CompanyId"
    ],
    "ProductionPlanning": [
        "Id",
        "MachineId",
        "FGQuantity",
        "FGCode",
        "FGName",
        "RMCode",
        "RMName",
        "RMQuantity",
        "QuantityInMT",
        "Color",
        "PackagingType",
        "InstructionForOperation",
        "MarketingRemarks",
        "RemarksByProduction",
        "Date",
        "InputQuantity",
        "InputDate",
        "NumberOfBags",
        "KGInPerBag",
        "CreatedBy",
        "CreatedOn",
        "UpdatedOn",
        "UpdatedBy",
        "CompanyId",
        "QRCodeUrl",
        "Status",
        "RMBagName",
        "MachineCode"
    ],
    "ProductMaster": [
        "ProductId",
        "ProductCode",
        "ProductName",
        "ProductType",
        "UOM",
        "Description",
        "StandardCost",
        "LeadTimeDays",
        "SafetyStock",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "PurchaseReturn": [
        "Id",
        "TransactionID",
        "PONumber",
        "PartyInvoiceNo",
        "PartyInvoiceDate",
        "ItemName",
        "ItemCode",
        "ReturnQty",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "PurchaseReturnDate",
        "NumberOfBags"
    ],
    "RawMaterial": [
        "RawMaterialId",
        "RawMaterialName",
        "PackagingType",
        "RawMaterialGrade",
        "RawMaterialWeight",
        "VendorId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "RawMaterial_FG_SFGMapping": [
        "RawMaterialId",
        "FGName",
        "SFGName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "IsActive"
    ],
    "Rejected": [
        "Id",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "RejectedQuantity",
        "UOM",
        "NumberOfBags",
        "KGInPerBags",
        "OutputDate",
        "Status",
        "StoreId",
        "QRCodeUrl",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "BagName",
        "Color",
        "ReasonForRepackaging",
        "RepackDate"
    ],
    "Repackaging": [
        "Id",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "NumberOfBags",
        "KGInPerBag",
        "BagName",
        "Remarks",
        "Date",
        "CompanyId",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "CreatedOn",
        "CustomerName",
        "UOM",
        "Color",
        "QRCodeUrl",
        "SONumber",
        "Status",
        "IsActive",
        "FGId"
    ],
    "ReturnData": [
        "ReturnId",
        "MaterialCode",
        "MaterialName",
        "QuantityReturned",
        "NumberOfBags",
        "KGInPerBag",
        "BagName",
        "Remarks",
        "Date",
        "CompanyId",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "CreatedOn",
        "CustomerName",
        "UOM",
        "Color",
        "SONumber",
        "DispatchNumber",
        "DispatchQuantity",
        "FGId",
        "IsActive"
    ],
    "RMGrade": [
        "RMGradeId",
        "RMGradeName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "RMInvoice": [
        "PONumber",
        "ItemCode",
        "ItemName",
        "BillRate",
        "BillBaseQuantity",
        "ReceivedQuantity",
        "InvoiceNumber",
        "InvoiceDate",
        "Price",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "SendToERP",
        "SendOn"
    ],
    "RMInward": [
        "Id",
        "PONumber",
        "ItemCode",
        "ItemName",
        "BillBaseQuantity",
        "RMGradeId",
        "InvoiceNumber",
        "InvoiceDate",
        "VehicleNumber",
        "PendingQuantity",
        "BatchId",
        "Specification",
        "ReceivedQuantity",
        "ReceivedQuantityUnit",
        "LotNumber",
        "RMInwardDate",
        "Price",
        "NumberOfBags",
        "KGInPerBags",
        "Status",
        "StoreId",
        "StoreAssignedDate",
        "MaterialConditions",
        "QRCodeUrl",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "RejectedQuantity",
        "ApprovedQuantity",
        "AvailableQuantity",
        "ApprovedNumberOfBags",
        "RejectedNumberOfBags",
        "ReturnQuantity",
        "RMHoldQuantity",
        "HoldNumberOfBags",
        "BillRate",
        "OrderRate"
    ],
    "RMInwardDetails": [
        "Id",
        "RequestId",
        "ItemCode",
        "ItemName",
        "QuantityInPerBag",
        "QRCodeUrl",
        "IsScanned",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "BagName",
        "Comments",
        "Category",
        "Status",
        "IsPriority"
    ],
    "RoutingMaster": [
        "RoutingId",
        "ProductId",
        "RoutingCode",
        "RoutingName",
        "Version",
        "TotalTimeMinutes",
        "Status",
        "IsActive",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "OperationSeq",
        "WorkCenterId",
        "StdCycleTime"
    ],
    "RoutingStep": [
        "RoutingStepId",
        "RoutingTemplateId",
        "Seq",
        "Operation",
        "WorkCenter",
        "Machine",
        "SetupTime",
        "RunTime",
        "WaitTime",
        "TotalTime",
        "Skill",
        "Status",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "RoutingTemplate": [
        "RoutingTemplateId",
        "TemplateCode",
        "TemplateName",
        "ProductCode",
        "Version",
        "Status",
        "CustomerId",
        "CreatedDate",
        "UpdatedDate",
        "CreatedBy",
        "UpdatedBy"
    ],
    "SalesForecastHeader": [
        "Id",
        "SalePersonId",
        "MonthYear",
        "Status",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "CustomerId"
    ],
    "SalesForecasts": [
        "Id",
        "RequestId",
        "CustomerCode",
        "MaterialCode",
        "MonthYear",
        "AutoCalculateQuantity",
        "ForecastQuantity",
        "PreviousMonthQuantity",
        "LastYearQuantity",
        "AdjustedQuantity",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "Price",
        "Amount",
        "WasEdited"
    ],
    "SalesOrder": [
        "SalesOrderId",
        "SONumber",
        "CustomerCode",
        "MaterialCode",
        "OrderQuantity",
        "UOM",
        "OrderDate",
        "Status",
        "TotalAmount",
        "Discount",
        "Tax",
        "NetAmount",
        "PaymentMethod",
        "ShippingAddress",
        "BillingAddress",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "IsActive",
        "PlannedDispatchDate",
        "ReasonForDateChanged"
    ],
    "SalesPerson": [
        "SalesPersonId",
        "FullName",
        "EmployeeCode",
        "Email",
        "PhoneNumber",
        "AlternatePhone",
        "Address",
        "CityId",
        "StateId",
        "CountryId",
        "PostalCode",
        "TargetSales",
        "AchievedSales",
        "ManagerId",
        "Designation",
        "TotalSales",
        "PerformanceRating",
        "AnnualBonus",
        "IsActive",
        "CreatedOn",
        "CreatedBy",
        "UpdatedOn",
        "UpdatedBy",
        "CompanyId"
    ],
    "SalesPersonCustomerMapping": [
        "SalesPersonId",
        "CustomerId",
        "CustomerCode",
        "IsActive",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "CompanyId"
    ],
    "SalesPersonQtyBlocked": [
        "ID",
        "SalespersonId",
        "SalespersonName",
        "BlockedQunatity",
        "MaterialCode",
        "MaterialName",
        "CompanyId",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "IsReleased",
        "CreatedOn"
    ],
    "SampleDetails": [
        "Id",
        "CustomerCode",
        "MaterialCode",
        "Quantity",
        "MonthYear",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "SalesPersonId",
        "Price",
        "Amount",
        "SampleSentOn"
    ],
    "Scrap": [
        "Id",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "ScrapQuantity",
        "UOM",
        "NumberOfBagsScrap",
        "KGInPerBagsScrap",
        "OutputDate",
        "Status",
        "StoreId",
        "PackagingType",
        "QRCodeUrl",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "BagName",
        "Color"
    ],
    "ScrapDispatch": [
        "Id",
        "RequestId",
        "VehicleNumber",
        "VehicleType",
        "NumberOfBags",
        "TotalQuantity",
        "KGInPerBag",
        "UOM",
        "Status",
        "DispatchDate",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "CreatedOn",
        "CompanyId",
        "QRCodeUrl"
    ],
    "SemiFinishedGood": [
        "SFGId",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "SFGQuantity",
        "OutputDate",
        "UOM",
        "NumberOfBags",
        "KGInPerBags",
        "StoreId",
        "ToleranceLevelId",
        "MachineId",
        "RawMaterialId",
        "Status",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "QRCodeUrl",
        "BagName",
        "Color",
        "ReasonForRepackaging",
        "RepackDate"
    ],
    "ServiceType": [
        "ServiceTypeId",
        "ServiceTypeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "ShiftCalendar": [
        "ShiftCalendarId",
        "Date",
        "ShiftId",
        "MachineId",
        "IsWorkingDay",
        "IsHoliday",
        "HolidayName",
        "AvailableHours",
        "CustomerId",
        "DayType",
        "Remarks",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "ShiftMaster": [
        "ShiftId",
        "ShiftCode",
        "ShiftName",
        "StartTime",
        "EndTime",
        "BreakMinutes",
        "IsActive",
        "CustomerId",
        "WorkCenterCode",
        "OperatorCount",
        "Status",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate",
        "Operators"
    ],
    "SpareType": [
        "SpareTypeId",
        "SpareTypeName",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn"
    ],
    "Store": [
        "StoreId",
        "StoreName",
        "PlantId",
        "YardId",
        "LocationId",
        "StoreManagerId",
        "InWardManagerId",
        "ManagerId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "TentativeRMPlan": [
        "Id",
        "Segment",
        "RMGrade",
        "RequirementInMT",
        "StockInHand",
        "MinimumStockLevel",
        "MaterialInTransit",
        "FinalRequirement",
        "MonthYear",
        "WeekNumber",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "ToleranceLevel": [
        "ToleranceLevelId",
        "ToleranceName",
        "MinValue",
        "MaxValue",
        "UnitOfMeasure",
        "Description",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "ToolingMaster": [
        "ToolId",
        "ToolCode",
        "ToolName",
        "ToolType",
        "AssignedToMachineCode",
        "AssignedToMachineId",
        "LifeUsed",
        "LifeTotal",
        "LastChangeDate",
        "NextChangeDate",
        "Status",
        "IsActive",
        "CustomerId",
        "CreatedDate",
        "UpdatedDate",
        "UpdatedBy",
        "CreatedBy"
    ],
    "UnPack": [
        "Id",
        "RequestId",
        "MaterialCode",
        "MaterialName",
        "Quantity",
        "NumberOfBags",
        "KGInPerBag",
        "BagName",
        "Remarks",
        "CompanyId",
        "UpdatedBy",
        "UpdatedOn",
        "CreatedBy",
        "CreatedOn",
        "UOM",
        "Color",
        "SONumber",
        "Status",
        "Date"
    ],
    "VehicleDetails": [
        "Id",
        "RequestId",
        "VehicleNumber",
        "VehicleType",
        "MaterialCode",
        "MaterialName",
        "Status",
        "LoadQuantity",
        "UnloadQuantity",
        "LoadedBags",
        "UnloadedBags",
        "KGInPerBag",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "FGId"
    ],
    "Vendors": [
        "VendorId",
        "VendorName",
        "PhoneNumber",
        "EmailAddress",
        "VendorCode",
        "Address",
        "VehicleNumber",
        "VendorCIN",
        "VendorGST",
        "CompanyWebsite",
        "IsActive",
        "CreatedBy",
        "CreatedOn",
        "UpdatedOn",
        "UpdatedBy",
        "PackagingTypeId",
        "CompanyId"
    ],
    "WeeklyPlanning": [
        "Id",
        "MaterialCode",
        "MonthYear",
        "WeekNumber",
        "Next10DaysRequirement",
        "RequiredProduction",
        "QtyInStock",
        "PlannedQty",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId"
    ],
    "WeighingMachine": [
        "Id",
        "FGQuantity",
        "SFGQuantity",
        "RejectedQuantity",
        "ScrapQuantity",
        "OnlineHoldQuantity",
        "MachineCode",
        "CreatedOn",
        "IsActive",
        "ShiftName",
        "ShiftStartTime",
        "ShiftEndTime"
    ],
    "WIPRMRequest": [
        "Id",
        "ItemCode",
        "RequestedQuantity",
        "UOM",
        "ReleasedQuantity",
        "ReceivedQuantity",
        "StoreId",
        "StoreName",
        "CreatedBy",
        "CreatedOn",
        "UpdatedOn",
        "UpdatedBy",
        "CompanyId",
        "Status"
    ],
    "WIPStorage": [
        "Id",
        "ParentRequestId",
        "MachineCode",
        "RMCode",
        "RMName",
        "FGCode",
        "FGName",
        "Remarks",
        "FGQuantity",
        "UOM",
        "RMQuantity",
        "InputDate",
        "InputQuantity",
        "Status",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "CompanyId",
        "QRCodeUrl",
        "FGNumberOfBags",
        "KGInPerBag",
        "SFGQuantity",
        "SFGNumberOfBags",
        "ScrapQuantity",
        "ScrapNumberOfBags",
        "RejectedQuantity",
        "RejectedNumberOfBags",
        "OnlineHoldQuantity",
        "OnlineHoldNumberOfBags",
        "SFGCode",
        "SFGName",
        "FinishedQuantity",
        "NumberOfBags",
        "Color",
        "RMBagName",
        "CurrentMachineStep",
        "Machine1Name",
        "Machine1InputQty",
        "Machine1OutputQty",
        "Machine2Name",
        "Machine2InputQty",
        "Machine2OutputQty",
        "Machine3Name",
        "Machine3InputQty",
        "Machine2Bins",
        "Machine3Bins",
        "MachineId",
        "ItemCode",
        "RMGradeId",
        "RequestedQuantity",
        "IssuedQuantity",
        "IssueDate",
        "RemainingQuantity",
        "QuantityInMachine",
        "OutputDate"
    ],
    "WIPStorageLocation": [
        "WIPStorageLocationId",
        "ItemName",
        "ItemCode",
        "NumberOfBags",
        "KGInPerBag",
        "ReceivedQuantity",
        "ReceivedDate",
        "UOM",
        "Remarks",
        "CompanyId",
        "CreatedBy",
        "CreatedOn",
        "UpdatedBy",
        "UpdatedOn",
        "RemainingQuantity",
        "BagName",
        "ProductionPlanningRequestId"
    ],
    "WorkflowActionTemplate": [
        "StepId",
        "ActionId",
        "ActionName",
        "NextStep",
        "TargetStepId",
        "NotificationTemplate",
        "SendNotification",
        "CloseWorkflow",
        "WarningMessage",
        "CommentRequired",
        "AttachmentRequired",
        "AttachmentMandatory",
        "WorkflowType"
    ],
    "WorkflowAgentsTemplate": [
        "StepId",
        "AgentGroupName",
        "Id",
        "IsActive",
        "SLADays",
        "WorkflowType"
    ],
    "WorkflowHistory": [
        "Id",
        "RequestId",
        "WorkStepId",
        "WorkStepName",
        "WorkStepDescription",
        "Remarks",
        "ActionTakenBy",
        "ActionDate",
        "CurrentStatus",
        "CurrentStatusName",
        "ActionName",
        "ActionComment",
        "ActionById"
    ],
    "WorkflowStepsTemplate": [
        "StepId",
        "Id",
        "FormUrl",
        "StepName",
        "StepDescription",
        "Active",
        "LocationId",
        "SLADays",
        "WorkflowType"
    ],
    "WorkOrder": [
        "WorkOrderId",
        "WorkOrderNumber",
        "MpsId",
        "ProductId",
        "PlannedQty",
        "CompletedQty",
        "UOM",
        "DueDate",
        "PriorityId",
        "RoutingId",
        "BomId",
        "MachineId",
        "OperatorId",
        "ShiftId",
        "PlannedStart",
        "PlannedEnd",
        "ActualStart",
        "ActualEnd",
        "PlannedDurationHours",
        "ActualDurationHours",
        "Status",
        "CurrentStep",
        "ProgressPercent",
        "HasException",
        "ExceptionCount",
        "CustomerId",
        "CustomerOrderRef",
        "BomVersion",
        "ReleasedDate",
        "ReleasedBy",
        "CancelReasonCode",
        "CancelReasonNotes",
        "CancelledDate",
        "Notes",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "WorkOrderBOM": [
        "WorkOrderBOMId",
        "WorkOrderId",
        "ComponentId",
        "Quantity",
        "UOM",
        "IssuedQty",
        "ConsumedQty",
        "ScrapQty",
        "Status",
        "CustomerId",
        "AllocatedQty",
        "RequiredQty",
        "ComponentProductId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "WorkOrderChangeLog": [
        "ChangeLogId",
        "WorkOrderId",
        "ChangeType",
        "FieldChanged",
        "OldValue",
        "NewValue",
        "ChangeReason",
        "ChangedBy",
        "ChangedAt",
        "CustomerId"
    ],
    "WorkOrderLog": [
        "WorkOrderLogId",
        "WorkOrderId",
        "EntryType",
        "Severity",
        "Title",
        "Description",
        "Author",
        "IsSystemGenerated",
        "Timestamp",
        "IsResolved",
        "ResolvedAt",
        "ResolvedBy",
        "ResolutionNote",
        "AlertId",
        "CustomerId",
        "CreatedDate",
        "CreatedBy",
        "UpdatedDate",
        "UpdatedBy"
    ],
    "WorkOrderResource": [
        "WorkOrderResourceId",
        "WorkOrderId",
        "ResourceType",
        "ResourceId",
        "StepSequence",
        "HoursCommitted",
        "Shift",
        "Status",
        "CustomerId",
        "CreatedDate",
        "CreatedBy",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "WorkOrderStep": [
        "WorkOrderStepId",
        "WorkOrderId",
        "RoutingStepId",
        "Sequence",
        "StepName",
        "OperationCode",
        "Status",
        "WorkcenterId",
        "OperatorId",
        "PlannedDurationMins",
        "ActualDurationMins",
        "PlannedQty",
        "CompletedQty",
        "ProgressPercent",
        "InstructionUrl",
        "StartedAt",
        "CompletedAt",
        "CustomerId",
        "CreatedBy",
        "CreatedDate",
        "UpdatedBy",
        "UpdatedDate"
    ],
    "WorkStepActions": [
        "WorkStepId",
        "ActionName",
        "RequestId",
        "ActionId",
        "NotificationTemplate",
        "TargetStepId",
        "NextWorkstepId",
        "SendNotification",
        "CloseWorkflow",
        "WarningMessage",
        "CommentRequired",
        "AttachmentRequired",
        "AttachmentMandatory"
    ],
    "WorkStepAgents": [
        "RequestId",
        "WorkstepId",
        "AgentGroupName",
        "Id",
        "SendNotification",
        "SLADays"
    ],
    "Worksteps": [
        "RequestId",
        "WorkstepId",
        "Id",
        "ParentRequestId",
        "WorkstepName",
        "WorkStepDescription",
        "InitiationDate",
        "FormUrl",
        "InitiatorId",
        "InitiatorName",
        "IsActive",
        "IsComplete",
        "Remarks",
        "ActionDate",
        "ActionBy",
        "ActionByName",
        "Action",
        "CurrentStatus",
        "CurrentStatusName",
        "SLADays",
        "AttachmentIds",
        "CurrentAction"
    ]
}

SCHEMA_CONTEXT = "\n".join(
    f"  [{tbl}]: {', '.join(cols[:15])}"
    for tbl, cols in MES_CATALOG.items()
)

print(f"✅ MES Schema Catalog loaded — {len(MES_CATALOG)} tables exposed for read-only access.")


In [ ]:
# CELL 5: System Agent Tools — execute_read_only_sql, get_mes_schema_details, list_available_tables
BLOCKED_SQL_KEYWORDS = [
    "DROP", "DELETE", "UPDATE", "INSERT", "ALTER",
    "TRUNCATE", "EXEC", "EXECUTE", "CREATE", "MERGE",
    "GRANT", "REVOKE", "DENY", "BACKUP", "RESTORE", "XP_", "SP_"
]

@tool
def execute_read_only_sql(query: str) -> str:
    """
    Executes a read-only SELECT query against the MES SQL Server / SQLite database.
    Use this tool to query any MES manufacturing table (WorkOrder, Inventory, RawMaterial, Machine, Alerts, etc.).
    Returns JSON formatted results (max 50 rows). ONLY SELECT queries are allowed.
    """
    if engine is None:
        return "Database engine not connected. Standard schema validation passed."
    upper_q = query.upper().strip()
    for kw in BLOCKED_SQL_KEYWORDS:
        if kw in upper_q:
            return f"SECURITY EXCEPTION: keyword '{kw}' is not allowed. Only SELECT queries are permitted."
    if not upper_q.startswith("SELECT"):
        return "SECURITY EXCEPTION: Only SELECT queries are permitted."
    try:
        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn)
        if df.empty:
            return "Query ran successfully — no matching records found."
        result_json = df.head(50).to_json(orient="records", date_format="iso", indent=2)
        note = f"\n[Showing 50 of {len(df)} rows]" if len(df) > 50 else ""
        return result_json + note
    except Exception as e:
        return f"SQL Execution Error: {e}\nPlease revise the query."

@tool
def get_mes_schema_details(table_name: str) -> str:
    """
    Returns the allowed columns for a specific MES database table.
    Use BEFORE writing SQL when you are unsure about exact column names.
    Example: get_mes_schema_details("WorkOrder")
    """
    clean_name = table_name.replace("[", "").replace("]", "").replace("dbo.", "").strip()
    if clean_name in MES_CATALOG:
        cols = ", ".join(MES_CATALOG[clean_name])
        return f"[dbo].[{clean_name}] has columns: {cols}"
    # Fuzzy match
    matched = [t for t in MES_CATALOG if clean_name.lower() in t.lower()]
    if matched:
        cols = ", ".join(MES_CATALOG[matched[0]])
        return f"Did you mean [dbo].[{matched[0]}]? Columns: {cols}"
    available = ", ".join(list(MES_CATALOG.keys())[:20])
    return f"Table '{table_name}' not found. First 20 available tables: {available}..."

@tool
def list_available_tables() -> str:
    """
    Lists all available MES database tables the System Agent can query.
    """
    table_list = ", ".join(MES_CATALOG.keys())
    return f"Available MES tables ({len(MES_CATALOG)}): {table_list}"

# Register system_tools
system_tools     = [execute_read_only_sql, get_mes_schema_details, list_available_tables]
system_tool_node = ToolNode(system_tools)

print("✅ System Agent tools registered:")
for t in system_tools:
    print(f"   • {t.name}")


In [ ]:
# CELL 6: Agent State + Agent Definitions (Deva Orchestrator & System Agent)
class AgentState(TypedDict):
    messages:     Annotated[Sequence[BaseMessage], operator.add]
    active_agent: str

GENERAL_SYSTEM_PROMPT = """You are **Deva**, the Manufacturing Intelligence Orchestrator for a Smart IIoT MES platform.
Your role:
1. Understand the user's manufacturing query (production, work orders, inventory, raw materials, vendors, Gantt, OEE, scrap, alerts, etc.).
2. If real-time data from the MES database is required, include the phrase [SYSTEM_AGENT] in your response along with the data specification.
3. Once the System Agent returns data, synthesize a clean executive summary, key KPIs, anomalies/trends, and recommendations.
Rules:
- NEVER write SQL yourself — delegate to System Agent.
- Be precise, professional, and data-driven.
"""

def general_orchestrator(state: AgentState) -> dict:
    sys_msg  = SystemMessage(content=GENERAL_SYSTEM_PROMPT)
    messages = [sys_msg] + list(state["messages"])
    response = invoke_with_fallback(
        orchestrator_llm_primary,
        orchestrator_llm_fallback,
        messages,
        role="orchestrator"
    )
    return {"messages": [response], "active_agent": "general"}

SYSTEM_AGENT_PROMPT = f"""You are the MES System Data Agent — a strictly-scoped database interface with READ-ONLY access.
Responsibilities:
1. Use `get_mes_schema_details` or `list_available_tables` to check column & table names.
2. Use `execute_read_only_sql` to execute SELECT queries.
3. Return raw results to Deva.
HARD RULES:
- Only SELECT queries allowed.
- Use TOP 50 for row limiting.
Available Schema Context:
{SCHEMA_CONTEXT[:3000]}
"""

def system_data_agent(state: AgentState) -> dict:
    sys_msg  = SystemMessage(content=SYSTEM_AGENT_PROMPT)
    messages = [sys_msg] + list(state["messages"])
    response = invoke_with_fallback(
        system_llm_primary,
        system_llm_fallback,
        messages,
        role="system",
        tools=system_tools
    )
    return {"messages": [response], "active_agent": "system"}

print("✅ Agents defined: General (Deva) + System Data Agent with system_tools.")


In [ ]:
# CELL 7: Graph Routing Logic
_DELEGATION_TRIGGERS = [
    "[system_agent]", "fetch data", "query the", "retrieve from", "database",
    "from the db", "select from", "check the table", "look up", "get records",
    "pull data", "data shows", "query database", "fetch from", "stock", "inventory",
    "work order", "machine", "vendor", "scrap", "alert", "po number", "boms"
]

def route_from_general(state: AgentState) -> str:
    last_msg = state["messages"][-1]
    content  = (getattr(last_msg, "content", "") or "").lower()
    if any(trigger in content for trigger in _DELEGATION_TRIGGERS):
        return "system"
    return END

def route_from_system(state: AgentState) -> str:
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "general"

print("✅ Graph routing functions defined.")


In [ ]:
# CELL 8: Build & Compile the LangGraph
builder = StateGraph(AgentState)
builder.add_node("general", general_orchestrator)
builder.add_node("system",  system_data_agent)
builder.add_node("tools",   system_tool_node)

builder.add_edge(START, "general")
builder.add_conditional_edges("general", route_from_general, {"system": "system", END: END})
builder.add_conditional_edges("system", route_from_system, {"tools": "tools", "general": "general"})
builder.add_edge("tools", "system")

mes_agent_app = builder.compile()
print("✅ MES Multi-Agent Graph compiled successfully!")


In [ ]:
# CELL 9: Interactive Query Runner Helper
def run_mes_query(user_question: str, verbose: bool = True) -> str:
    sep  = "=" * 70
    dash = "-" * 70
    print(f"\n{sep}")
    print("  MANUFACTURING AI — MES QUERY (DEVA & SYSTEM AGENT)")
    print(f"  User Question: {user_question}")
    print(dash)
    initial_state = {"messages": [HumanMessage(content=user_question)], "active_agent": "general"}
    final_answer = ""
    icons = {"general": "[GENERAL/DEVA]", "system": "[SYSTEM/DB]", "tools": "[TOOL EXEC]"}
    for event in mes_agent_app.stream(initial_state, {"recursion_limit": 15}):
        for node_name, state_update in event.items():
            if "messages" not in state_update: continue
            latest_msg = state_update["messages"][-1]
            if verbose:
                label = icons.get(node_name, f"[{node_name.upper()}]")
                print(f"\n  {label}")
                if hasattr(latest_msg, "tool_calls") and latest_msg.tool_calls:
                    for tc in latest_msg.tool_calls:
                        print(f"    >> Tool called: {tc['name']}")
                        print(f"       Args: {str(tc.get('args', ''))[:250]}")
                elif isinstance(latest_msg, ToolMessage):
                    print(f"    >> DB Result (preview): {(latest_msg.content or '')[:300]}...")
                elif hasattr(latest_msg, "content") and latest_msg.content:
                    print(f"    >> {latest_msg.content[:500]}")
            if node_name == "general" and hasattr(latest_msg, "content"):
                final_answer = latest_msg.content
    print(f"\n{sep}")
    print("  FINAL ANSWER FROM DEVA")
    print(sep)
    print(final_answer)
    print(f"{sep}\n")
    return final_answer

print("✅ run_mes_query() helper ready.")


In [ ]:
# CELL 10: Run Sample Query
user_question = "What is the current stock of raw material RM-102?"
answer = run_mes_query(user_question, verbose=True)


In [ ]:
# CELL 11: 100 Manufacturing Sample Queries
# ────────────────────────────────────────────────────────────────────────────
# Uncomment any of the 100 manufacturing queries below and run this cell!

SAMPLE_100_QUERIES = [
    "What is the current stock of raw material RM-102?",
    "Show me all raw materials below their minimum stock level.",
    "How much finished goods inventory do we have for product code FG-505?",
    "List all material batches currently on QA hold.",
    "What is the total scrap quantity generated this week?",
    "Show me the inventory history for lot number LOT-890.",
    "Which warehouse rack location holds barcode batch B-778?",
    "Get the list of rejected raw materials from the last 30 days.",
    "What is the total quantity of semi-finished goods waiting for processing?",
    "Display the stock ledger for item category 'Plastics'.",
    "List all pending purchase orders for supplier 'ABC Plastics'.",
    "What was the received quantity for PO number PO-2023-45?",
    "Show me the invoice details for RM inward request RM-00012.",
    "Which incoming vehicles are currently at the gate waiting for unloading?",
    "Are there any discrepancies between billed quantity and received quantity today?",
    "Get the list of unapproved inward batches.",
    "Show me all RM inwards that have been assigned to Store A.",
    "What is the average price of RM grade 'A' purchased this month?",
    "List all purchase returns made in the last quarter.",
    "Which vendor has the highest rejection rate for raw materials?",
    "What is the Bill of Materials for product FG-100?",
    "Are there any active Master Production Schedules for next week?",
    "Show me the MRP run results for the latest planning horizon.",
    "What raw material shortages are predicted for the upcoming MPS?",
    "List all work orders scheduled for Machine-01 today.",
    "What is the current progress percentage of Work Order WO-998?",
    "Are there any overlapping schedules on the Gantt chart for the injection molding machines?",
    "Show me the weekly production plan for material code M-123.",
    "What is the projected stock for product P-456 after all planned orders are completed?",
    "Get the routing steps required to manufacture product code FG-200.",
    "Which machines are currently marked as 'Offline'?",
    "Show me the capacity utilization for Machine-05 yesterday.",
    "Who is the operator assigned to shift 'Morning' on Machine-02?",
    "List all machines that are due for maintenance this week.",
    "What is the total production output for Shift A today?",
    "Are there any capacity overloads flagged for tomorrow's shift?",
    "Show me the live OEE stats for the packaging line.",
    "What were the actual vs planned hours for Work Order WO-555?",
    "List all active work order exceptions or stoppages.",
    "Which work centers have the lowest efficiency percentage this month?",
    "What is the status of Sales Order SO-1001?",
    "List all sales orders that are planned for dispatch today.",
    "How many finished goods bags have been loaded onto vehicle DL-1CA-1234?",
    "Are there any discrepancies in the dispatch scan for order SO-1002?",
    "Show me the total sales forecast for customer 'XYZ Corp' next month.",
    "Which items are currently in transit?",
    "List all sales persons who have not met their target sales this quarter.",
    "What is the dispatch history for customer code CUST-099?",
    "Show me the jumbo packaging details for sales order SO-1005.",
    "Which batches were allocated to dispatch request DIS-0004?",
    "List all open breakdown maintenance requests.",
    "What is the current status of work order request WR-204?",
    "Show me the maintenance history for Machine-03.",
    "Which tools are currently checked out by maintenance technicians?",
    "What is the remaining life of tool code T-500?",
    "Are there any scheduled maintenance windows for the extrusion machines tomorrow?",
    "Show me the repair logs added by technician John Doe.",
    "List all spare parts consumed in maintenance work orders this month.",
    "What is the average resolution time for high-severity breakdown alerts?",
    "Show me the tool assignment history for Work Order WO-777.",
    "Which QA auditor approved RM inward batch B-123?",
    "Show me the workflow history for approval request REQ-999.",
    "List all materials that failed tolerance level checks today.",
    "What are the rejection comments for batch B-404?",
    "Show me all pending tasks in my workflow queue.",
    "Which finished goods batches are currently marked as 'Scrap'?",
    "How many online hold requests were generated during the night shift?",
    "What is the average scrap percentage for routing step 'Molding'?",
    "Show me the repackaging reasons logged this week.",
    "List all alerts generated with severity 'Critical'.",
    "What are the active locations defined under Plant-01?",
    "List all users with the 'PRODUCTION_MANAGER' role.",
    "Show me the master configuration for Item Group 'Resins'.",
    "What is the default storage location for material M-001?",
    "List all registered vendors for packaging materials.",
    "Show me the shift calendar details for the upcoming public holiday.",
    "What are the defined priority levels for maintenance requests?",
    "List all active customers in the 'North' region.",
    "What is the base conversion ratio for unit 'Box' to 'PCS'?",
    "Show me the HSN code mapped to finished product FG-300.",
    "Find the PO number associated with invoice INV-9999.",
    "What is the exact rack coordinate for WIP storage request WIP-0005?",
    "Did we receive any barcode scanning errors on the dispatch dock today?",
    "Show me the bill of materials version history for product P-222.",
    "Calculate the total required quantity of RM-10 for next week's MPS.",
    "Who acknowledged the machine breakdown alert on line 2?",
    "What is the cycle time defined for MachineFGMapping ID 10?",
    "List all products that have a lead time greater than 5 days.",
    "What is the maximum capacity for Machine 'Extruder-A'?",
    "Show me the plastic category and material type for RM-55.",
    "How many cavities are configured for mold MOLD-01?",
    "What was the previous month's sales forecast quantity for customer C-12?",
    "Are there any work order logs containing the keyword 'overheating'?",
    "List the workflow agents assigned to the QA approval step.",
    "What is the standard cost of producing one unit of FG-88?",
    "Show me the tentative RM plan for segment 'Retail' in week 4.",
    "Which store manager is assigned to 'Main RM Store'?",
    "Find all finished goods batches that have been repacked more than once.",
    "What is the exact timestamp when Work Order WO-100 was released?",
    "Give me a summary of total dispatched vs total produced quantities for this month."
]

print(f"✅ {len(SAMPLE_100_QUERIES)} Manufacturing sample queries loaded.")
# Example run on query #1:
# run_mes_query(SAMPLE_100_QUERIES[0])


In [ ]:
# CELL 12: Function to Check MES Schema from schema.sql File
# ────────────────────────────────────────────────────────────────────────────
def check_mes_schema_from_sql_file(schema_file_path: str = r"c:\Users\renuk\OneDrive\Desktop\MANUFACTURING AGENTIC AI\Manufacturing-Agentic-Ai-iiiot\docs\mes_db_info\schema.sql") -> dict:
    """
    Parses an MES DB schema.sql file to extract table names and their column lists.
    Returns a dictionary of {table_name: [column_names]} and updates the global MES_CATALOG.
    """
    import os, re
    global MES_CATALOG, SCHEMA_CONTEXT
    if not os.path.exists(schema_file_path):
        print(f"⚠️ Schema file not found at: {schema_file_path}")
        return MES_CATALOG
    with open(schema_file_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    matches = re.findall(r"CREATE TABLE\s+\[dbo\]\.\[(\w+)\]\s*\((.*?)\)\s*ON", content, re.DOTALL)
    parsed_catalog = {}
    for tbl, cols_block in matches:
        cols = []
        for line in cols_block.splitlines():
            line = line.strip()
            m = re.match(r"\[(\w+)\]\s+\[(\w+)\]", line)
            if m:
                col_name = m.group(1)
                if col_name.upper() not in ("CONSTRAINT", "PRIMARY", "KEY", "FOREIGN", "REFERENCES"):
                    cols.append(col_name)
        if cols:
            parsed_catalog[tbl] = list(dict.fromkeys(cols))
    if parsed_catalog:
        MES_CATALOG.update(parsed_catalog)
        SCHEMA_CONTEXT = "\n".join(f"  [{tbl}]: {', '.join(cols[:15])}" for tbl, cols in MES_CATALOG.items())
        print(f"✅ Successfully parsed {len(parsed_catalog)} tables from {os.path.basename(schema_file_path)}.")
        print(f"   Total MES_CATALOG tables now: {len(MES_CATALOG)}")
    return parsed_catalog

# Run schema check function
cat_result = check_mes_schema_from_sql_file()
